# Image Quality Score — PSF Moments Across the Focal Plane

This notebook builds on the dataset collected in `psf_moments_allbands.ipynb`.

## Image quality definition

A visit has **good image quality** when:
- PSF size ($\sigma$) is small
- Ellipticity modulus $|e|$ is small
- Coma modulus $|\mathrm{coma}|$ is small
- Trefoil modulus $|\mathrm{trefoil}|$ is small
- All of the above vary **little** across the focal plane

The composite score is a weighted sum of percentile ranks of these 8 sub-scores
(4 FOV-medians + 4 focal-plane standard deviations).  
Low score = good quality.

### Weighting rationale
| Component | Weight | Reasoning |
|---|---|---|
| σ (FOV median) | 0.21 | Fundamental: sets resolution limit |
| σ (FP scatter) | 0.14 | Focal-plane uniformity of PSF size |
| \|e\| (FOV median) | 0.15 | Critical for weak-lensing shape meas. |
| \|e\| (FP scatter) | 0.10 | Ellipticity uniformity |
| \|coma\| (FOV median) | 0.12 | Breaks circular symmetry of PSF |
| \|coma\| (FP scatter) | 0.08 | |
| \|trefoil\| (FOV median) | 0.12 | Third-order aberration |
| \|trefoil\| (FP scatter) | 0.08 | |

Each sub-score is the **percentile rank** within the full combined dataset,
so the composite score lives in [0, 1] with 0 = best, 1 = worst.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import rankdata

%matplotlib inline

PIXEL_SCALE = 1.0   # arcsec/pixel
DATA_DIR    = "/sdf/data/rubin/user/ztq1996/psf-rubin/psf_zernike/data"

SCIENCE_RAFTS = [
    "R01","R02","R03",
    "R10","R11","R12","R13","R14",
    "R20","R21","R22","R23","R24",
    "R30","R31","R32","R33","R34",
    "R41","R42","R43",
]

ZERNIKE_COLS = ["z4","z5","z6","z7","z8","z9","z10","z11"]
ZERNIKE_LABELS = [
    "Z4 (defocus)","Z5 (astig 45°)","Z6 (astig 0°)",
    "Z7 (coma x)","Z8 (coma y)",
    "Z9 (trefoil x)","Z10 (trefoil y)",
    "Z11 (spherical)",
]

BAND_COLORS = {"u":"#7b2d8b","g":"#1a9641","r":"#d73027",
               "i":"#fc8d59","z":"#4575b4","y":"#8c510a"}

## 1. Load dataset

In [ ]:
combined = pd.read_parquet(f"{DATA_DIR}/psf_moments_allbands.pq")
print(f"Loaded {len(combined):,} visit-rows across bands: {sorted(combined['band'].unique())}")
# band = "g"
# combined = combined[combined['band']==band]
combined.head(3)

## 2. Derive scalar quantities

For each visit compute:
- **FOV-median** scalar magnitudes: $\sigma$, $|e|$, $|\mathrm{coma}|$, $|\mathrm{trefoil}|$
- **Focal-plane scatter** (std over rafts) of those same quantities

In [ ]:
plt.hist(df['dimm_seeing'], bins = 100)
plt.show()

In [ ]:
df = combined.copy()

# ── FOV-median scalars ───────────────────────────────────────────────────────
df["sigma"]    = np.sqrt(df["T"] / 2) * PIXEL_SCALE #- df['dimm_seeing']/2.5          # arcsec
df["fwhm"]    = np.sqrt(df["T"] / 2) * PIXEL_SCALE * 2.3548 #- df['dimm_seeing']/2.5          # arcsec
df["mod_e"]    = np.sqrt(df["e1"]**2  + df["e2"]**2)
df["mod_coma"] = np.sqrt(df["c11"]**2 + df["c12"]**2)
df["mod_tref"] = np.sqrt(df["c31"]**2 + df["c32"]**2)

# ── Per-raft sigma, |e|, |coma|, |trefoil| ──────────────────────────────────
for raft in SCIENCE_RAFTS:
    T_col  = f"T_{raft}"
    e1_col, e2_col   = f"e1_{raft}",  f"e2_{raft}"
    c11_col, c12_col = f"c11_{raft}", f"c12_{raft}"
    c31_col, c32_col = f"c31_{raft}", f"c32_{raft}"

    if T_col in df.columns:
        df[f"sigma_{raft}"] = np.sqrt(df[T_col] / 2) * PIXEL_SCALE
        df[f"fwhm_{raft}"] = np.sqrt(df[T_col] / 2) * PIXEL_SCALE * 2.3548 
    if e1_col in df.columns and e2_col in df.columns:
        df[f"mod_e_{raft}"] = np.sqrt(df[e1_col]**2 + df[e2_col]**2)
        df[f"e1_{raft}"] = df[e1_col]
        df[f"e2_{raft}"] = df[e2_col]
    if c11_col in df.columns and c12_col in df.columns:
        df[f"mod_coma_{raft}"] = np.sqrt(df[c11_col]**2 + df[c12_col]**2)
    if c31_col in df.columns and c32_col in df.columns:
        df[f"mod_tref_{raft}"] = np.sqrt(df[c31_col]**2 + df[c32_col]**2)

# ── Focal-plane scatter (std over rafts per visit) ───────────────────────────
sigma_raft_cols    = [f"sigma_{r}"    for r in SCIENCE_RAFTS if f"sigma_{r}"    in df.columns]
fwhm_raft_cols    = [f"fwhm_{r}"    for r in SCIENCE_RAFTS if f"fwhm_{r}"    in df.columns]
mod_e_raft_cols    = [f"mod_e_{r}"    for r in SCIENCE_RAFTS if f"mod_e_{r}"    in df.columns]
e1_raft_cols    = [f"e1_{r}"    for r in SCIENCE_RAFTS if f"e1_{r}"    in df.columns]
e2_raft_cols    = [f"e2_{r}"    for r in SCIENCE_RAFTS if f"e2_{r}"    in df.columns]
mod_coma_raft_cols = [f"mod_coma_{r}" for r in SCIENCE_RAFTS if f"mod_coma_{r}" in df.columns]
mod_tref_raft_cols = [f"mod_tref_{r}" for r in SCIENCE_RAFTS if f"mod_tref_{r}" in df.columns]

df["sigma_fp_std"]    = df[sigma_raft_cols].std(axis=1)
df["fwhm_fp_std"]    = df[fwhm_raft_cols].std(axis=1)
df["mod_e_fp_std"]    = df[mod_e_raft_cols].std(axis=1)
df["e1_fp_std"]    = df[e1_raft_cols].std(axis=1)
df["e2_fp_std"]    = df[e2_raft_cols].std(axis=1)
df["mod_coma_fp_std"] = df[mod_coma_raft_cols].std(axis=1)
df["mod_tref_fp_std"] = df[mod_tref_raft_cols].std(axis=1)

print("Derived FOV-median scalars and focal-plane scatter.")
df[["sigma","sigma_fp_std","mod_e","mod_e_fp_std",
    "mod_coma","mod_coma_fp_std","mod_tref","mod_tref_fp_std"]].describe()

## 3. Compute image quality score

In [ ]:
def percentile_rank(series: pd.Series) -> pd.Series:
    """Return percentile rank in [0, 1]: 0 = smallest, 1 = largest.
    NaNs are ranked last (worst) and then masked back to NaN.
    """
    vals   = series.values.copy()
    finite = np.isfinite(vals)
    ranks  = np.full(len(vals), np.nan)
    n      = finite.sum()
    if n > 0:
        ranks[finite] = (rankdata(vals[finite], method="average") - 1) / (n - 1)
    return pd.Series(ranks, index=series.index)


def compute_image_quality(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add an `image_quality` column to *df* (lower = better).

    Sub-scores (each in [0,1], 0 = best):
        sigma_rank        : percentile rank of FOV-median PSF size
        sigma_fp_rank     : percentile rank of FP scatter of PSF size
        mod_e_rank        : percentile rank of FOV-median |e|
        mod_e_fp_rank     : percentile rank of FP scatter of |e|
        mod_coma_rank     : percentile rank of FOV-median |coma|
        mod_coma_fp_rank  : percentile rank of FP scatter of |coma|
        mod_tref_rank     : percentile rank of FOV-median |trefoil|
        mod_tref_fp_rank  : percentile rank of FP scatter of |trefoil|

    Weights (sum = 1.00):
        PSF size          : 0.35  (0.21 mean + 0.14 scatter)
        Ellipticity       : 0.25  (0.15 mean + 0.10 scatter)
        Coma              : 0.20  (0.12 mean + 0.08 scatter)
        Trefoil           : 0.20  (0.12 mean + 0.08 scatter)
    """
    out = df.copy()

    sub_scores = {
        "fwhm_rank":       ("fwhm",            0.3),
        "fwhm_fp_rank":    ("fwhm_fp_std",     0.14),
        "mod_e_rank":       ("mod_e",            0.15),
        "mod_e_fp_rank":    ("mod_e_fp_std",     0.10),
        "mod_coma_rank":    ("mod_coma",         0.12),
        "mod_coma_fp_rank": ("mod_coma_fp_std",  0.08),
        "mod_tref_rank":    ("mod_tref",         0.12),
        "mod_tref_fp_rank": ("mod_tref_fp_std",  0.08),
    }

    # sub_scores = {
    #     "sigma_rank":       ("sigma",            0.3),
    #     "sigma_fp_rank":    ("sigma_fp_std",     0.1),
    #     "mod_e_rank":       ("mod_e",            0.3),
    #     "mod_e_fp_rank":    ("mod_e_fp_std",     0.1),
    #     "mod_coma_rank":    ("mod_coma",         0.1),
    #     "mod_coma_fp_rank": ("mod_coma_fp_std",  0.05),
    #     "mod_tref_rank":    ("mod_tref",         0.1),
    #     "mod_tref_fp_rank": ("mod_tref_fp_std",  0.05),
    # }

    score = pd.Series(np.zeros(len(out)), index=out.index)
    for rank_col, (src_col, weight) in sub_scores.items():
        out[rank_col] = percentile_rank(out[src_col])
        score += weight * out[rank_col].fillna(1.0)   # NaN penalised as worst

    out["image_quality"] = score
    return out


df = compute_image_quality(df)
print("image_quality stats:")
print(df["image_quality"].describe())

### Quick look: image quality distribution per band

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for band, grp in df.groupby("band"):
    ax.hist(grp["image_quality"].dropna(), bins=50, histtype="step",
            label=band, color=BAND_COLORS.get(band, "grey"), linewidth=1.5)
ax.set_xlabel("Image quality score  (0 = best, 1 = worst)", fontsize=11)
ax.set_ylabel("Visits", fontsize=11)
ax.set_title("Image quality score distribution per band", fontsize=12)
ax.legend(title="band")
plt.tight_layout()
plt.show()

### Sub-score contributions

In [ ]:
rank_cols = ["fwhm_rank","fwhm_fp_rank",
             "mod_e_rank","mod_e_fp_rank",
             "mod_coma_rank","mod_coma_fp_rank",
             "mod_tref_rank","mod_tref_fp_rank"]
weights   = [0.21, 0.14, 0.15, 0.10, 0.12, 0.08, 0.12, 0.08]
short_labels = [
    r"FWHM (med)", r"FWHM (FP std)",
    r"$|e|$ (med)",    r"$|e|$ (FP std)",
    r"|coma| (med)",   r"|coma| (FP std)",
    r"|trefoil| (med)",r"|trefoil| (FP std)",
]

fig, ax = plt.subplots(figsize=(10, 4))
medians = df[rank_cols].median().values
bars = ax.bar(short_labels, [w * m for w, m in zip(weights, medians)],
              color=["#2166ac","#92c5de"]*4)
ax.set_ylabel("Weighted median contribution to score", fontsize=10)
ax.set_title("Sub-score contributions to image_quality (median over all visits)", fontsize=11)
ax.tick_params(axis="x", labelsize=9)
plt.tight_layout()
plt.show()

## 4. Best and worst 20 % by image quality

In [ ]:
q20 = df["image_quality"].quantile(0.20)
q80 = df["image_quality"].quantile(0.80)

df_best  = df[df["image_quality"] <= q20].copy()
df_worst = df[df["image_quality"] >= q80].copy()
df_all   = df.copy()

print(f"Best  20%: {len(df_best):,} visits  (score ≤ {q20:.3f})")
print(f"Worst 20%: {len(df_worst):,} visits  (score ≥ {q80:.3f})")
print(f"All       : {len(df_all):,} visits")

### Band composition of best / worst subsets

In [ ]:
bands = sorted(df["band"].unique())
counts_all   = df["band"].value_counts().reindex(bands, fill_value=0)
counts_best  = df_best["band"].value_counts().reindex(bands, fill_value=0)
counts_worst = df_worst["band"].value_counts().reindex(bands, fill_value=0)

x = np.arange(len(bands))
w = 0.25
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w, counts_all   / counts_all.sum()   * 100, w, label="All",     color="grey",     alpha=0.7)
ax.bar(x,     counts_best  / counts_best.sum()  * 100, w, label="Best 20%", color="#2166ac",  alpha=0.85)
ax.bar(x + w, counts_worst / counts_worst.sum() * 100, w, label="Worst 20%",color="#d73027",  alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(bands, fontsize=11)
ax.set_ylabel("Fraction of subset [%]", fontsize=10)
ax.set_title("Band composition — all / best / worst 20%", fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()

### PSF scalar distributions: best / worst / all

In [ ]:
scalar_specs = [
    ("fwhm",    r"FWHM [arcsec]"),
    ("mod_e",    r"$|e|$"),
    ("mod_coma", r"$|\mathrm{coma}|$"),
    ("mod_tref", r"$|\mathrm{trefoil}|$"),
    ("sigma_fp_std",    r"$\sigma$ FP std [arcsec]"),
    ("mod_e_fp_std",    r"$|e|$ FP std"),
    ("mod_coma_fp_std", r"$|\mathrm{coma}|$ FP std"),
    ("mod_tref_fp_std", r"$|\mathrm{trefoil}|$ FP std"),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
fig.suptitle(f"PSF scalar distributions — all / best 20% / worst 20% | {band}-band", fontsize=13, fontweight="bold")

for ax, (col, label) in zip(axes.flat, scalar_specs):
    lo = np.nanpercentile(df_all[col].dropna(), 0.1)
    hi = np.nanpercentile(df_all[col].dropna(), 99.9)
    bins = np.linspace(lo, hi, 50)
    ax.hist(df_all[col].dropna(),   bins=bins, density=True, histtype="stepfilled",
            alpha=0.3, color="grey",    label="All")
    ax.hist(df_best[col].dropna(),  bins=bins, density=True, histtype="step",
            linewidth=2, color="#2166ac", label="Best 20%")
    ax.hist(df_worst[col].dropna(), bins=bins, density=True, histtype="step",
            linewidth=2, color="#d73027", label="Worst 20%")
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel("Density", fontsize=8)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## 5. Zernike coefficient histograms: best 20% vs all visits

For each Zernike Z4–Z11 we overlay:
- **All visits** (grey filled)
- **Best 20%** (blue step)

In [ ]:
def zernike_comparison_plot(df_all, df_subset, subset_label, subset_color,
                             title, ncols=4):
    """Overlay Zernike histograms of df_subset (step) on df_all (filled)."""
    nrows = int(np.ceil(len(ZERNIKE_COLS) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    for ax, zc, zlabel in zip(axes.flat, ZERNIKE_COLS, ZERNIKE_LABELS):
        all_vals = df_all[zc].dropna()
        sub_vals = df_subset[zc].dropna()

        lo = np.percentile(all_vals, 0.1)
        hi = np.percentile(all_vals, 99.9)
        bins = np.linspace(lo, hi, 55)

        ax.hist(all_vals, bins=bins, density=True, histtype="stepfilled",
                color="grey", alpha=0.4, label=f"All  (N={len(all_vals):,})")
        ax.hist(sub_vals, bins=bins, density=True, histtype="step",
                color=subset_color, linewidth=2.0,
                label=f"{subset_label}  (N={len(sub_vals):,})")

        # Mark medians
        ax.axvline(np.median(all_vals), color="grey",        linestyle="--", linewidth=1.2)
        ax.axvline(np.median(sub_vals), color=subset_color,  linestyle="--", linewidth=1.5)

        ax.set_title(zlabel, fontsize=9)
        ax.set_xlabel("Zernike coefficient [nm]", fontsize=8)
        ax.set_ylabel("Density", fontsize=8)
        ax.legend(fontsize=7)

    # Hide any spare axes
    for ax in axes.flat[len(ZERNIKE_COLS):]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.show()


zernike_comparison_plot(
    df_all, df_best, "Best 20%", "#2166ac",
    title="Zernike coefficients — Best 20% image quality vs all visits"
)

## 6. Zernike coefficient histograms: worst 20% vs all visits

In [ ]:
zernike_comparison_plot(
    df_all, df_worst, "Worst 20%", "#d73027",
    title="Zernike coefficients — Worst 20% image quality vs all visits"
)

## 7. Side-by-side: best, all, worst — all Zernikes in one figure

In [ ]:
ncols = 4
nrows = int(np.ceil(len(ZERNIKE_COLS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
fig.suptitle(
    f"Zernike coefficients — Best 20% (blue) / All (grey) / Worst 20% (red) | {band}-band",
    fontsize=12, fontweight="bold"
)

for ax, zc, zlabel in zip(axes.flat, ZERNIKE_COLS, ZERNIKE_LABELS):
    all_vals   = df_all[zc].dropna()
    best_vals  = df_best[zc].dropna()
    worst_vals = df_worst[zc].dropna()

    lo = np.percentile(all_vals, 0.1)
    hi = np.percentile(all_vals, 99.9)
    bins = np.linspace(lo, hi, 55)

    ax.hist(all_vals,   bins=bins, density=True, histtype="stepfilled",
            color="grey",    alpha=0.35, label=f"All ({len(all_vals):,})")
    ax.hist(best_vals,  bins=bins, density=True, histtype="step",
            color="#2166ac", linewidth=2.0, label=f"Best 20% ({len(best_vals):,})")
    ax.hist(worst_vals, bins=bins, density=True, histtype="step",
            color="#d73027", linewidth=2.0, label=f"Worst 20% ({len(worst_vals):,})")

    # Median markers
    ax.axvline(np.median(all_vals),   color="grey",    linestyle=":",  linewidth=1.2)
    ax.axvline(np.median(best_vals),  color="#2166ac", linestyle="--", linewidth=1.5)
    ax.axvline(np.median(worst_vals), color="#d73027", linestyle="--", linewidth=1.5)

    ax.set_title(zlabel, fontsize=9)
    ax.set_xlabel("Zernike coefficient [um]", fontsize=8)
    ax.set_ylabel("Density", fontsize=8)
    ax.legend(fontsize=6.5)

for ax in axes.flat[len(ZERNIKE_COLS):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

## 8. Summary statistics: Zernike medians and spreads

In [ ]:
rows = []
for zc, zlabel in zip(ZERNIKE_COLS, ZERNIKE_LABELS):
    for subset_label, subset_df in [("All", df_all), ("Best 20%", df_best), ("Worst 20%", df_worst)]:
        vals = subset_df[zc].dropna()
        rows.append({
            "Zernike": zlabel,
            "Subset":  subset_label,
            "N":       len(vals),
            "Median":  np.median(vals),
            "MAD":     np.median(np.abs(vals - np.median(vals))),
            "Std":     vals.std(),
        })

summary = pd.DataFrame(rows)
pivot_median = summary.pivot(index="Zernike", columns="Subset", values="Median")
pivot_std    = summary.pivot(index="Zernike", columns="Subset", values="Std")

print("=== Zernike medians [nm] ===")
print(pivot_median[["Best 20%", "All", "Worst 20%"]].to_string(float_format="{:.3f}".format))
print()
print("=== Zernike std [nm] ===")
print(pivot_std[["Best 20%", "All", "Worst 20%"]].to_string(float_format="{:.3f}".format))

## 9. Zernike spread comparison: best vs worst 20%

Bar chart of the **std** of each Zernike for best / all / worst, showing how much tighter
the Zernike distributions are for high-quality visits.

In [ ]:
std_best  = [df_best[z].std()  for z in ZERNIKE_COLS]
std_all   = [df_all[z].std()   for z in ZERNIKE_COLS]
std_worst = [df_worst[z].std() for z in ZERNIKE_COLS]

x = np.arange(len(ZERNIKE_COLS))
w = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w, std_all,   w, label="All",      color="grey",    alpha=0.7)
ax.bar(x,     std_best,  w, label="Best 20%",  color="#2166ac", alpha=0.85)
ax.bar(x + w, std_worst, w, label="Worst 20%", color="#d73027", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(ZERNIKE_LABELS, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Std of Zernike coefficient [nm]", fontsize=10)
ax.set_title("Zernike coefficient spread — all / best 20% / worst 20%", fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.hist(df['image_quality'], bins = 100)
plt.show()

In [ ]:
df.to_parquet('/sdf/data/rubin/user/ztq1996/psf-rubin/psf_zernike/data/dp2_all_visits_psf+zernike_with_image_quality.pq')

In [ ]:
df.columns

In [ ]:
# q20 = df["image_quality"].quantile(0.20)
q80 = df["image_quality"].quantile(0.80)

df_best  = df[df["image_quality"] <= q80].copy()
df_worst = df[df["image_quality"] >= q80].copy()
df_all   = df.copy()

print(f"Best  80%: {len(df_best):,} visits  (score ≤ {q80:.3f})")
print(f"Worst 20%: {len(df_worst):,} visits  (score ≥ {q80:.3f})")
print(f"All       : {len(df_all):,} visits")

In [ ]:
scalar_specs = [
    ("fwhm",    r"FWHM [arcsec]"),
    ("mod_e",    r"$|e|$"),
    ("mod_coma", r"$|\mathrm{coma}|$"),
    ("mod_tref", r"$|\mathrm{trefoil}|$"),
    ("sigma_fp_std",    r"$\sigma$ FP std [arcsec]"),
    ("mod_e_fp_std",    r"$|e|$ FP std"),
    ("mod_coma_fp_std", r"$|\mathrm{coma}|$ FP std"),
    ("mod_tref_fp_std", r"$|\mathrm{trefoil}|$ FP std"),
]


density = False

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
fig.suptitle(f"PSF scalar distributions — all / best 80% / worst 20% | {band}-band", fontsize=13, fontweight="bold")

for ax, (col, label) in zip(axes.flat, scalar_specs):
    lo = np.nanpercentile(df_all[col].dropna(), 0.1)
    hi = np.nanpercentile(df_all[col].dropna(), 99.9)
    bins = np.linspace(lo, hi, 50)
    ax.hist(df_all[col].dropna(),   bins=bins, density=density, histtype="stepfilled",
            alpha=0.3, color="grey",    label=f"All | mean:{round(np.mean(df_all[col]),3)},\n median:{round(np.nanmedian(df_all[col]),3)}, std:{round(np.std(df_all[col]),3)} ")
    ax.hist(df_best[col].dropna(),  bins=bins, density=density, histtype="step",
            linewidth=2, color="#2166ac", label=f"Best 80% | mean:{round(np.mean(df_best[col]),3)},\n median:{round(np.nanmedian(df_best[col]),3)}, std:{round(np.std(df_best[col]),3)} ")
    ax.hist(df_worst[col].dropna(), bins=bins, density=density, histtype="step",
            linewidth=2, color="#d73027", label=f"Worst 20% | mean:{round(np.mean(df_worst[col]),3)},\n median:{round(np.nanmedian(df_worst[col]),3)}, std:{round(np.std(df_worst[col]),3)} ")
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel("Counts", fontsize=8)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
df['e1_fp_std']

In [ ]:
scalar_specs = [
    ("fwhm",    r"FWHM [arcsec]"),
    ("e1",    r"$e_1$"),
    ("e2", r"$e_2$"),
    ("mod_tref", r"$|\mathrm{trefoil}|$"),
    ("fwhm_fp_std",    r"FWHM FP std [arcsec]"),
    ("e1_fp_std",    r"$e_1$ FP std"),
    ("e2_fp_std",    r"$e_2$ FP std"),
]


density = False

fig, axes = plt.subplots(2, 3, figsize=(18, 7))
fig.suptitle(f"PSF scalar distributions — all / best 80% / worst 20% | {band}-band", fontsize=13, fontweight="bold")

for ax, (col, label) in zip(axes.flat, scalar_specs):
    lo = np.nanpercentile(df_all[col].dropna(), 0.1)
    hi = np.nanpercentile(df_all[col].dropna(), 99.9)
    bins = np.linspace(lo, hi, 50)
    ax.hist(df_all[col].dropna(),   bins=bins, density=density, histtype="stepfilled",
            alpha=0.3, color="grey",    label=f"All | mean:{round(np.mean(df_all[col]),3)},\n median:{round(np.nanmedian(df_all[col]),3)}, std:{round(np.std(df_all[col]),3)} ")
    ax.hist(df_best[col].dropna(),  bins=bins, density=density, histtype="step",
            linewidth=2, color="#2166ac", label=f"Best 80% | mean:{round(np.mean(df_best[col]),3)},\n median:{round(np.nanmedian(df_best[col]),3)}, std:{round(np.std(df_best[col]),3)} ")
    ax.hist(df_worst[col].dropna(), bins=bins, density=density, histtype="step",
            linewidth=2, color="#d73027", label=f"Worst 20% | mean:{round(np.mean(df_worst[col]),3)},\n median:{round(np.nanmedian(df_worst[col]),3)}, std:{round(np.std(df_worst[col]),3)} ")
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel("Counts", fontsize=8)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
ncols = 4
nrows = int(np.ceil(len(ZERNIKE_COLS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
fig.suptitle(
    f"Zernike coefficients — Best 20% (blue) / All (grey) / Worst 20% (red) | {band}-band",
    fontsize=12, fontweight="bold"
)

for ax, zc, zlabel in zip(axes.flat, ZERNIKE_COLS, ZERNIKE_LABELS):
    all_vals   = df_all[zc].dropna()
    best_vals  = df_best[zc].dropna()
    worst_vals = df_worst[zc].dropna()

    lo = np.percentile(all_vals, 0.1)
    hi = np.percentile(all_vals, 99.9)
    bins = np.linspace(lo, hi, 55)

    ax.hist(all_vals,   bins=bins, density=density, histtype="stepfilled",
            color="grey",    alpha=0.35, label=f"All ({len(all_vals):,})")
    ax.hist(best_vals,  bins=bins, density=density, histtype="step",
            color="#2166ac", linewidth=2.0, label=f"Best 80% ({len(best_vals):,})")
    ax.hist(worst_vals, bins=bins, density=density, histtype="step",
            color="#d73027", linewidth=2.0, label=f"Worst 20% ({len(worst_vals):,})")

    # Median markers
    ax.axvline(np.median(all_vals),   color="grey",    linestyle=":",  linewidth=1.2)
    ax.axvline(np.median(best_vals),  color="#2166ac", linestyle="--", linewidth=1.5)
    ax.axvline(np.median(worst_vals), color="#d73027", linestyle="--", linewidth=1.5)


    ax.set_title(zlabel, fontsize=9)
    ax.set_xlabel("Zernike coefficient [um]", fontsize=8)
    ax.set_ylabel("Counts", fontsize=8)
    ax.legend(fontsize=6.5)

for ax in axes.flat[len(ZERNIKE_COLS):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.colors import LogNorm

plt.hist2d( df_all['mod_coma'], df_all['sigma'], bins = 30, range = ((0,0.1),(0.4,0.8)), norm = LogNorm())
plt.show()

plt.hist2d( df_all['mod_e'], df_all['sigma'], bins = 30, range = ((0,0.2),(0.4,0.8)), norm = LogNorm())
plt.show()

## 10. PSF distributions by human annotation

Cross-match with `data/annotations.csv` (per-detector labels: 0=good, 0.5=okay, 1=bad).
Annotations are aggregated per visit by rounding the mean detector label to the nearest 0.5.

In [ ]:
np.unique(ann['visit'])

In [ ]:
ann['detector']

In [ ]:
# ── Load and aggregate annotations ──────────────────────────────────────────
ann = pd.read_csv(f"{DATA_DIR}/annotations.csv")

ann = ann[ann['annotation']>-1]

# Aggregate per visit: mean annotation rounded to nearest 0.5 → {0, 0.5, 1}
ann_visit = (
    ann.groupby("visit")["annotation"]
    .mean()
    .apply(lambda x: round(x * 2) / 2)
    .reset_index()
    .rename(columns={"visit": "visit_id", "annotation": "ann_label"})
)

# Cross-match with main df
df_ann = df.merge(ann_visit, on="visit_id", how="inner")
print(f"Annotated visits matched: {len(df_ann):,}  "
      f"(good={( df_ann['ann_label']==0).sum()}, "
      f"okay={(df_ann['ann_label']==0.5).sum()}, "
      f"bad={(df_ann['ann_label']==1).sum()})")

df_good = df_ann[df_ann["ann_label"] == 0.0]
df_okay = df_ann[df_ann["ann_label"] == 0.5]
df_bad  = df_ann[df_ann["ann_label"] == 1.0]

ANN_COLORS = {0.0: "#2166ac", 0.5: "#fdae61", 1.0: "#d73027"}
ANN_LABELS = {0.0: "Good (0)", 0.5: "Okay (0.5)", 1.0: "Bad (1)"}

# ── Plot ─────────────────────────────────────────────────────────────────────
scalar_specs_ann = [
    ("fwhm",          r"FWHM [arcsec]"),
    ("mod_e",          r"$|e|$"),
    ("mod_coma",       r"$|\mathrm{coma}|$"),
    ("mod_tref",       r"$|\mathrm{trefoil}|$"),
    ("sigma_fp_std",   r"$\sigma$ FP std [arcsec]"),
    ("mod_e_fp_std",   r"$|e|$ FP std"),
    ("mod_coma_fp_std",r"$|\mathrm{coma}|$ FP std"),
    ("mod_tref_fp_std",r"$|\mathrm{trefoil}|$ FP std"),
]

density = False

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
fig.suptitle(
    f"PSF scalar distributions by annotation | {band}-band",
    fontsize=13, fontweight="bold"
)

for ax, (col, label) in zip(axes.flat, scalar_specs_ann):
    lo = np.nanpercentile(df_ann[col].dropna(), 0.1)
    hi = np.nanpercentile(df_ann[col].dropna(), 99.9)
    bins = np.linspace(lo, hi, 40)

    for ann_val, subset in [(0.0, df_good), (0.5, df_okay), (1.0, df_bad)]:
        vals = subset[col].dropna()
        if len(vals) == 0:
            continue
        lbl = (
            f"{ANN_LABELS[ann_val]} | "
            f"mean:{np.mean(vals):.3f},\n"
            f" med:{np.nanmedian(vals):.3f}, std:{np.std(vals):.3f}"
        )
        ax.hist(
            vals, bins=bins, density=density,
            histtype="step" if ann_val > 0 else "stepfilled",
            alpha=0.5 if ann_val == 0 else 1.0,
            linewidth=2, color=ANN_COLORS[ann_val], label=lbl
        )

    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel("Counts", fontsize=8)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()